# SWOW Multi-Language Alpaca Format Dataset Generation

Converts the three SWOW datasets (Dutch `nl`, Rioplatense Spanish `rp`, Chinese `zh`) from
`data/01_raw/SWOW_SIMON/` into Alpaca-format JSONL training files ready for use with LLaMA-Factory.

**Data sources**
| Language | File |
|----------|------|
| Dutch (nl) | `SWOW-NL.2012.csv` |
| Spanish / Rioplatense (rp) | `SWOWRP.R70.20220426.csv` |
| Chinese (zh) | `SWOWZH.R55.20230424.processed.csv` |

**Output** → `data/03_primary/swow_multi_lang/{language}/seed_{n}/`  
Each seed folder contains `train/chunk_{25|50|75|100}_percent_train.jsonl`, `test.jsonl`, and `cue_meta_info.json`.

In [2]:
! pip install jsonlines pandas


  Using cached jsonlines-4.0.0-py3-none-any.whl.metadata (1.6 kB)
Using cached jsonlines-4.0.0-py3-none-any.whl (8.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 136.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 196.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [jsonlines]/4 [pandas]


In [3]:
import sys
import random
import jsonlines
from copy import deepcopy
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

# Make swow_gen_prompts importable (it lives in the same directory as this notebook)
NOTEBOOK_DIR = Path().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

RELATIVE_DIR = "notebooks/Simon-2-Dataset-Generation-For-Multi-lang"
if str(NOTEBOOK_DIR / RELATIVE_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR / RELATIVE_DIR))

from swow_gen_prompts import get_training_example

In [4]:
# project root ends with cultural-lexis-finetune-llms
PROJECT_ROOT = NOTEBOOK_DIR if NOTEBOOK_DIR.name == "cultural-lexis-finetune-llms" else NOTEBOOK_DIR.parent.parent

In [6]:
# ── Paths ─────────────────────────────────────────────────────────────────────

RAW_DATA_DIR = PROJECT_ROOT / "data" / "01_raw" / "SWOW_SIMON"
OUTPUT_BASE  = PROJECT_ROOT / "data" / "03_primary" / "swow_multi_lang"

NL_CSV = RAW_DATA_DIR / "SWOW-NL.2012.csv"
RP_CSV = RAW_DATA_DIR / "SWOWRP.R70.20220426.csv"
ZH_CSV = RAW_DATA_DIR / "SWOWZH.R55.20230424.processed.csv"

print("Project root :", PROJECT_ROOT)
print("Raw data dir :", RAW_DATA_DIR)
print("Output base  :", OUTPUT_BASE)
assert NL_CSV.exists(), f"Missing: {NL_CSV}"
assert RP_CSV.exists(), f"Missing: {RP_CSV}"
assert ZH_CSV.exists(), f"Missing: {ZH_CSV}"

Project root : /data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms
Raw data dir : /data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/data/01_raw/SWOW_SIMON
Output base  : /data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/data/03_primary/swow_multi_lang


## 1. Load raw CSV datasets

In [7]:
nl_df = pd.read_csv(NL_CSV)
rp_df = pd.read_csv(RP_CSV)
zh_df = pd.read_csv(ZH_CSV)

for name, df in [("NL", nl_df), ("RP", rp_df), ("ZH", zh_df)]:
    print(f"{name}: {len(df):>8,} rows  |  {df['cue'].nunique():>5,} unique cues"
          f"  |  R1 NA: {df['R1'].isna().sum():,}"
          f"  |  R2 NA: {df['R2'].isna().sum():,}"
          f"  |  R3 NA: {df['R3'].isna().sum():,}")

NL: 1,257,100 rows  |  12,571 unique cues  |  R1 NA: 17,892  |  R2 NA: 33,778  |  R3 NA: 72,892
RP:  944,159 rows  |  13,498 unique cues  |  R1 NA: 44,723  |  R2 NA: 279,588  |  R3 NA: 404,376
ZH:  550,044 rows  |  10,024 unique cues  |  R1 NA: 1  |  R2 NA: 0  |  R3 NA: 0


## 2. Convert rows to Alpaca training format

Each SWOW row contains `cue`, `R1`, `R2`, `R3`.  
`get_training_example` from `swow_gen_prompts.py` produces a dict with keys `system`, `instruction`, `input`, `output`.

**Missing-response tokens per language**

| Language | Token |
|----------|-------|
| nl | `NO MORE RESPONSE` |
| rp | `NO MÁS RESPUESTAS` |
| zh | `没有更多答案` |

In [8]:
# Language-specific "no more response" tokens (must match swow_gen_prompts.py templates)
MISSING_TOKEN = {
    'nl': 'NO MORE RESPONSE',
    'zh': '没有更多答案',
    'rp': 'NO MÁS RESPUESTAS',
}
# The ZH preprocessed CSV uses '#Missing' for politically-sensitive responses
MISSING_MARKER = '#Missing'


def clean_response(val, missing_tok: str) -> str:
    """Return val as a stripped string, substituting the language-specific
    missing token for NaN or the #Missing sentinel."""
    if pd.isna(val) or str(val).strip() == MISSING_MARKER:
        return missing_tok
    return str(val).strip()


def df_to_alpaca(df: pd.DataFrame, language: str) -> list[dict]:
    """Convert an entire SWOW DataFrame to a list of Alpaca-format dicts."""
    missing_tok = MISSING_TOKEN[language]
    examples = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"[{language.upper()}] converting"):
        r1 = clean_response(row['R1'], missing_tok)
        r2 = clean_response(row['R2'], missing_tok)
        r3 = clean_response(row['R3'], missing_tok)
        examples.append(get_training_example(language, str(row['cue']), r1, r2, r3))
    return examples


nl_alpaca = df_to_alpaca(nl_df, 'nl')
rp_alpaca = df_to_alpaca(rp_df, 'rp')
zh_alpaca = df_to_alpaca(zh_df, 'zh')

[NL] converting:   0%|          | 0/1257100 [00:00<?, ?it/s]

[RP] converting:   0%|          | 0/944159 [00:00<?, ?it/s]

[ZH] converting:   0%|          | 0/550044 [00:00<?, ?it/s]

In [9]:
# ── Filter out rows where R1 itself was missing (output starts with the token) ─
def remove_invalid(examples: list[dict], language: str) -> list[dict]:
    missing_tok = MISSING_TOKEN[language]
    valid = [ex for ex in examples if not ex['output'].startswith(missing_tok)]
    removed = len(examples) - len(valid)
    print(f"[{language.upper()}] {len(examples):,} → {len(valid):,}  (removed {removed:,} rows with missing R1)")
    return valid

nl_valid = remove_invalid(nl_alpaca, 'nl')
rp_valid = remove_invalid(rp_alpaca, 'rp')
zh_valid = remove_invalid(zh_alpaca, 'zh')

[NL] 1,257,100 → 1,239,208  (removed 17,892 rows with missing R1)
[RP] 944,159 → 899,436  (removed 44,723 rows with missing R1)
[ZH] 550,044 → 546,920  (removed 3,124 rows with missing R1)


In [10]:
# ── Quick sanity-check: show one example per language ──────────────────────────
for lang, data in [('nl', nl_valid), ('rp', rp_valid), ('zh', zh_valid)]:
    ex = data[0]
    print(f"=== {lang.upper()} ===")
    print(f"  input  : {ex['input']}")
    print(f"  output : {ex['output']}")
    print()

=== NL ===
  input  : Promptwoord: aubergine
  output : groente, paars, courgette

=== RP ===
  input  : Palabra clave: ?
  output : pregunta, NO MÁS RESPUESTAS, NO MÁS RESPUESTAS

=== ZH ===
  input  : 提示词: T恤
  output : 啼血, 球鞋, 短袖



## 3. Cluster by cue word and perform train / test split

Examples are clustered by cue word so that the same cue word never appears in both train and test splits.  
An 80 / 20 split is applied on the **unique cue words** using a fixed seed (`TRAIN_TEST_SEED = 1234`).

In [11]:
TRAIN_TEST_SEED = 1234


def cluster_by_cue(examples: list[dict]) -> dict[str, list[dict]]:
    """Group examples by their cue word (stored in the 'input' field)."""
    clusters: dict[str, list[dict]] = {}
    for ex in examples:
        key = ex['input']
        clusters.setdefault(key, []).append(ex)
    return clusters


def train_test_split_by_cue(
    clusters: dict[str, list[dict]],
    test_ratio: float = 0.2,
    seed: int = TRAIN_TEST_SEED,
) -> tuple[list[str], list[str]]:
    """Return (train_keys, test_keys) split by cue word."""
    keys = sorted(clusters.keys())
    rng = random.Random(seed)
    rng.shuffle(keys)
    split_idx = int(len(keys) * (1 - test_ratio))
    return keys[:split_idx], keys[split_idx:]


# ── Per-language clustering & splitting ───────────────────────────────────────
datasets: dict[str, dict] = {}
for lang, data in [('nl', nl_valid), ('rp', rp_valid), ('zh', zh_valid)]:
    clusters   = cluster_by_cue(data)
    train_keys, test_keys = train_test_split_by_cue(clusters)
    datasets[lang] = {
        'clusters':   clusters,
        'train_keys': train_keys,
        'test_keys':  test_keys,
    }
    print(f"[{lang.upper()}] unique cues: {len(clusters):,}"
          f"  |  train cues: {len(train_keys):,}"
          f"  |  test cues:  {len(test_keys):,}")

[NL] unique cues: 12,571  |  train cues: 10,056  |  test cues:  2,515
[RP] unique cues: 13,498  |  train cues: 10,798  |  test cues:  2,700
[ZH] unique cues: 10,024  |  train cues: 8,019  |  test cues:  2,005


## 4. Generate training chunks and save per-language JSONL files

For each of **5 random seeds**, the train cue words are shuffled and split into 4 incremental 25 % chunks:

| Chunk file | Cues included |
|------------|---------------|
| `chunk_25_percent_train.jsonl` | first 25 % of shuffled train cues |
| `chunk_50_percent_train.jsonl` | first 50 % |
| `chunk_75_percent_train.jsonl` | first 75 % |
| `chunk_100_percent_train.jsonl` | all train cues |

Output structure:
```
data/03_primary/swow_multi_lang/
  {language}/
    seed_{n}/
      train/
        chunk_25_percent_train.jsonl
        chunk_50_percent_train.jsonl
        chunk_75_percent_train.jsonl
        chunk_100_percent_train.jsonl
      test.jsonl
      cue_meta_info.json
```

In [12]:
SEED_LIST  = [42, 43, 44, 45, 46]
CHUNK_PCTS = [25, 50, 75, 100]   # incremental percentage checkpoints

IDX_TO_LABEL = {0: "25_percent_train",
                1: "50_percent_train",
                2: "75_percent_train",
                3: "100_percent_train"}


def save_jsonl(path: Path, records: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with jsonlines.open(path, mode='w') as writer:
        writer.write_all(records)


def generate_and_save(
    language: str,
    clusters: dict[str, list[dict]],
    train_keys: list[str],
    test_keys: list[str],
    output_dir: Path,
    seed_list: list[int] = SEED_LIST,
) -> None:
    """For each seed, shuffle train cues, cut into 4 cumulative chunks, and save."""
    # Build and save the test set once (same across seeds)
    test_data: list[dict] = []
    for key in test_keys:
        test_data.extend(clusters[key])

    for seed_idx, seed in enumerate(tqdm(seed_list, desc=f"[{language.upper()}] seeds")):
        seed_dir = output_dir / language / f"seed_{seed_idx + 1}"
        train_dir = seed_dir / "train"

        shuffled = deepcopy(train_keys)
        rng = random.Random(seed)
        rng.shuffle(shuffled)

        chunk_size = len(shuffled) // 4
        cue_meta = {
            "seed": seed,
            "train_test_seed": TRAIN_TEST_SEED,
            "cue_info": {
                "25%":  [],
                "50%":  [],
                "75%":  [],
                "100%": [],
                "test": test_keys,
            }
        }

        for chunk_idx in tqdm(range(4), leave=False, desc="chunks"):
            start = chunk_idx * chunk_size
            end   = (chunk_idx + 1) * chunk_size if chunk_idx < 3 else len(shuffled)
            chunk_keys = shuffled[start:end]
            pct_label  = f"{(chunk_idx + 1) * 25}%"
            cue_meta["cue_info"][pct_label] = chunk_keys

            # Cumulative: all cues up to (and including) current chunk
            cumulative_keys = shuffled[:end]
            chunk_data: list[dict] = []
            for k in cumulative_keys:
                chunk_data.extend(clusters[k])
            rng.shuffle(chunk_data)

            out_path = train_dir / f"chunk_{IDX_TO_LABEL[chunk_idx]}.jsonl"
            save_jsonl(out_path, chunk_data)

        # Save test set
        rng.shuffle(test_data)
        save_jsonl(seed_dir / "test.jsonl", test_data)

        # Save metadata
        meta_path = seed_dir / "cue_meta_info.json"
        meta_path.parent.mkdir(parents=True, exist_ok=True)
        import json
        with open(meta_path, 'w', encoding='utf-8') as f:
            json.dump(cue_meta, f, ensure_ascii=False, indent=2)

    print(f"[{language.upper()}] saved to {output_dir / language}")


# ── Run for NL, RP, ZH ────────────────────────────────────────────────────────
for lang in ['nl', 'rp', 'zh']:
    d = datasets[lang]
    generate_and_save(
        language   = lang,
        clusters   = d['clusters'],
        train_keys = d['train_keys'],
        test_keys  = d['test_keys'],
        output_dir = OUTPUT_BASE,
    )

[NL] seeds:   0%|          | 0/5 [00:00<?, ?it/s]

chunks:   0%|          | 0/4 [00:00<?, ?it/s]

chunks:   0%|          | 0/4 [00:00<?, ?it/s]

chunks:   0%|          | 0/4 [00:00<?, ?it/s]

chunks:   0%|          | 0/4 [00:00<?, ?it/s]

...

chunks:   0%|          | 0/4 [00:00<?, ?it/s]

chunks:   0%|          | 0/4 [00:00<?, ?it/s]

chunks:   0%|          | 0/4 [00:00<?, ?it/s]

chunks:   0%|          | 0/4 [00:00<?, ?it/s]

[ZH] saved to /data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/data/03_primary/swow_multi_lang/zh


## 5. (Optional) Combined multi-language dataset

Merges NL + RP + ZH into a single dataset, performing the same train/test split and chunking at the combined level.

In [ ]:
# Prefix each combined cue key with the language code to avoid cross-language
# cue collisions (e.g. 'nl::appel', 'zh::苹果').
combined_clusters: dict[str, list[dict]] = {}
for lang in ['nl', 'rp', 'zh']:
    for cue_key, examples in datasets[lang]['clusters'].items():
        prefixed_key = f"{lang}::{cue_key}"
        combined_clusters[prefixed_key] = examples

train_keys_combined, test_keys_combined = train_test_split_by_cue(combined_clusters)

print(f"Combined unique cues : {len(combined_clusters):,}")
print(f"Train cues           : {len(train_keys_combined):,}")
print(f"Test cues            : {len(test_keys_combined):,}")

generate_and_save(
    language   = 'combined',
    clusters   = combined_clusters,
    train_keys = train_keys_combined,
    test_keys  = test_keys_combined,
    output_dir = OUTPUT_BASE,
)

In [13]:
import json

# ── Summary ───────────────────────────────────────────────────────────────────
print("=== Output summary ===")
for lang in ['nl', 'rp', 'zh', 'combined']:
    lang_dir = OUTPUT_BASE / lang
    if not lang_dir.exists():
        continue
    jsonl_files = sorted(lang_dir.rglob("*.jsonl"))
    total_examples = 0
    for jf in jsonl_files:
        with jsonlines.open(jf) as r:
            n = sum(1 for _ in r)
        total_examples += n
    print(f"  {lang:<10}: {len(jsonl_files):>3} JSONL files  |  {total_examples:>10,} total examples across all files")
print(f"\nAll outputs saved under: {OUTPUT_BASE}")

=== Output summary ===
  nl        :  25 JSONL files  |  13,629,078 total examples across all files
  rp        :  25 JSONL files  |   9,892,709 total examples across all files
  zh        :  25 JSONL files  |   6,015,084 total examples across all files

All outputs saved under: /data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/data/03_primary/swow_multi_lang


## 6. Generate HuggingFace dataset `README.md`

The `README.md` (YAML frontmatter) tells HuggingFace Datasets how to load each split.  
One config entry is generated per **(language, seed, chunk-percentage)** triple, mirroring the pattern used in `data/03_primary/participant_swow_collection_us/README.md`.

| Config name pattern | Covers |
|---------------------|--------|
| `swow_multi_lang_{lang}_seed_{n}` | 100 % train + test for seed *n* |
| `swow_multi_lang_{lang}_seed_{n}_25pct` | 25 % train + test |
| `swow_multi_lang_{lang}_seed_{n}_50pct` | 50 % train + test |
| `swow_multi_lang_{lang}_seed_{n}_75pct` | 75 % train + test |

In [14]:
import yaml

# Chunk file stem  →  short tag used in config names
_CHUNK_TAG = {
    "25_percent_train":  "25pct",
    "50_percent_train":  "50pct",
    "75_percent_train":  "75pct",
    "100_percent_train": "full",   # no suffix appended for the "full" config
}


def build_hf_readme_configs(
    languages: list[str],
    seed_list: list[int],
    dataset_name: str = "swow_multi_lang",
) -> list[dict]:
    """Return a list of HuggingFace dataset config dicts (one per split variant).

    For every (language, seed) pair four configs are emitted:
      - full  -> chunk_100_percent_train.jsonl  (no suffix in config name)
      - 25pct -> chunk_25_percent_train.jsonl
      - 50pct -> chunk_50_percent_train.jsonl
      - 75pct -> chunk_75_percent_train.jsonl
    All configs include the matching test.jsonl as the test split.
    """
    configs = []
    for lang in languages:
        for seed_idx, _ in enumerate(seed_list):
            seed_label = f"seed_{seed_idx + 1}"
            base = f"{lang}/{seed_label}"

            for chunk_stem, tag in _CHUNK_TAG.items():
                config_name = (
                    f"{dataset_name}_{lang}_{seed_label}"
                    if tag == "full"
                    else f"{dataset_name}_{lang}_{seed_label}_{tag}"
                )
                configs.append({
                    "config_name": config_name,
                    "data_files": [
                        {"split": "train",
                         "path": f"{base}/train/chunk_{chunk_stem}.jsonl"},
                        {"split": "test",
                         "path": f"{base}/test.jsonl"},
                    ],
                })
    return configs


def generate_hf_readme(
    output_dir: Path,
    languages: list[str],
    seed_list: list[int],
    dataset_name: str = "swow_multi_lang",
) -> Path:
    """Write a HuggingFace-compatible README.md to *output_dir*.

    The file contains only the YAML front-matter block that `datasets` uses
    to discover splits, matching the style of
    ``data/03_primary/participant_swow_collection_us/README.md``.

    Parameters
    ----------
    output_dir:   Directory that contains the language sub-folders (OUTPUT_BASE).
    languages:    List of language codes present in output_dir.
    seed_list:    List of seed values (length determines seed_1..seed_N labels).
    dataset_name: Prefix used in every config_name.

    Returns
    -------
    Path to the written README.md.
    """
    configs  = build_hf_readme_configs(languages, seed_list, dataset_name)
    yaml_str = yaml.dump(
        {"configs": configs},
        allow_unicode=True,
        default_flow_style=False,
        sort_keys=False,
    )
    readme_content = f"---\n{yaml_str}---\n"

    readme_path = output_dir / "README.md"
    readme_path.parent.mkdir(parents=True, exist_ok=True)
    readme_path.write_text(readme_content, encoding="utf-8")

    print(f"README.md written to: {readme_path}")
    print(f"Total configs generated: {len(configs)}")
    return readme_path


# ── Generate ──────────────────────────────────────────────────────────────────
ALL_LANGUAGES = ["nl", "rp", "zh", "combined"]

readme_path = generate_hf_readme(
    output_dir   = OUTPUT_BASE,
    languages    = ALL_LANGUAGES,
    seed_list    = SEED_LIST,
    dataset_name = "swow_multi_lang",
)

# ── Preview first 60 lines ────────────────────────────────────────────────────
print("\n--- README.md preview (first 60 lines) ---\n")
lines = readme_path.read_text(encoding="utf-8").splitlines()
print("\n".join(lines[:60]))


README.md written to: /data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/data/03_primary/swow_multi_lang/README.md
Total configs generated: 80

--- README.md preview (first 60 lines) ---

---
configs:
- config_name: swow_multi_lang_nl_seed_1_25pct
  data_files:
  - split: train
    path: nl/seed_1/train/chunk_25_percent_train.jsonl
  - split: test
    path: nl/seed_1/test.jsonl
- config_name: swow_multi_lang_nl_seed_1_50pct
  data_files:
  - split: train
    path: nl/seed_1/train/chunk_50_percent_train.jsonl
  - split: test
    path: nl/seed_1/test.jsonl
- config_name: swow_multi_lang_nl_seed_1_75pct
  data_files:
  - split: train
    path: nl/seed_1/train/chunk_75_percent_train.jsonl
  - split: test
    path: nl/seed_1/test.jsonl
- config_name: swow_multi_lang_nl_seed_1
  data_files:
  - split: train
    path: nl/seed_1/train/chunk_100_percent_train.jsonl
  - split: test
    path: nl/seed_1/test.jsonl
- config_name: swow_multi_lang_nl_seed_2_25pct
  data_fi